# Lab 7 — Embedding and Detecting LLM Watermarks

*Statistical Foundations of LLMs · Session 4 · Accompanies slides 343–344 (Tracking AI-Generated Content, slide 342)*

How can a platform prove a piece of text was written by a specific AI model? **Watermarking** hides a statistical signal in the model's word choices at generation time; a **binomial hypothesis test** later detects it. In this lab you implement the influential **green-list watermark** (Kirchenbauer et al., 2023) from scratch and detect it with classical inference.


## Learning Objectives

1. Explain the **green-list / red-list** watermarking idea.
2. **Embed** a watermark by biasing a model's token logits at generation time.
3. **Detect** the watermark with a **binomial z-test** on the fraction of green tokens.
4. Quantify detectability, false-positive rate, and robustness to text edits.
5. Connect watermark detection to the statistical inference concepts from the previous lab.


## 0. New to Jupyter? Start Here (2 minutes)

**What is a Jupyter Notebook?** A document that mixes text and runnable Python code, organized in *cells*.

| What you need to know | How |
|---|---|
| **Run a cell** | Click it, then press **Shift + Enter** (or the ▶ button) |
| **Cell types** | **Markdown** cells = formatted text (like this one). **Code** cells = Python you can execute |
| **Order matters** | Run cells **top to bottom**. A cell may depend on variables defined above it |
| **Restart the kernel** | Menu: *Runtime → Restart runtime* (Colab) or *Kernel → Restart* (Jupyter). Then re-run cells from the top |
| **Install packages** | Run a cell starting with `%pip install ...`, then restart the kernel if asked |
| **Modify code** | Just edit any code cell and re-run it — experimenting is the whole point! |
| **Read outputs** | Results appear directly below each code cell: printed text, tables, or plots |

> 💡 **Tip:** If something behaves strangely, *Restart runtime* and run all cells from the top (*Runtime → Run all*).


## 1. Background: The Green-List Watermark

At each generation step, before sampling the next token, the watermark:

1. Uses the **previous token** to seed a random number generator.
2. Randomly splits the vocabulary into a **green list** (fraction $\gamma$) and a **red list** (the rest).
3. Adds a small bias $\delta$ to the logits of green-list tokens, making them more likely.

A human writer picks green vs. red tokens ~$\gamma$ of the time (chance). A watermarked model picks green tokens *much* more often. Detection is a **one-sided proportion test**:

$$z = \frac{|s|_G - \gamma T}{\sqrt{T\,\gamma(1-\gamma)}}$$

where $T$ is the number of tokens and $|s|_G$ the number that landed on the green list. Large $z$ ⇒ watermark present. This is a **binomial test** — exactly the classical inference from slide 359.


## 2. Setup and Imports


In [ ]:
# Run once if needed:
# %pip install -U transformers torch numpy scipy matplotlib --quiet


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import stats
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

set_seed(42)
np.random.seed(42)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
VOCAB_SIZE = model.config.vocab_size

# Watermark hyperparameters
GAMMA = 0.5   # fraction of vocabulary on the green list
DELTA = 2.5   # logit bias added to green tokens (0 = no watermark)
print(f"Vocab size {VOCAB_SIZE:,} | gamma={GAMMA} | delta={DELTA}")


## 3. The Green List: Seeded by the Previous Token

The green list must be *reproducible* at detection time, so we seed it deterministically from the previous token id.


In [ ]:
def green_list_ids(prev_token_id, gamma=GAMMA, vocab_size=VOCAB_SIZE):
    """Deterministically pick the green-list token ids given the previous token."""
    rng = np.random.default_rng(int(prev_token_id))          # same seed -> same split
    n_green = int(gamma * vocab_size)
    return set(rng.choice(vocab_size, size=n_green, replace=False).tolist())

g = green_list_ids(tokenizer.encode(" the")[0])
print(f"Green list has {len(g):,} tokens ({len(g)/VOCAB_SIZE:.0%} of vocab)")
print("Is ' movie' green?", tokenizer.encode(" movie")[0] in g)


## 4. Embed the Watermark During Generation

We generate token by token. At each step we add `DELTA` to the logits of green-list tokens, then sample. Setting `delta=0` recovers ordinary (unwatermarked) generation — handy for the control group.


In [ ]:
def generate(prompt, max_new_tokens=80, delta=DELTA, gamma=GAMMA, temperature=0.7, seed=42):
    """Generate text; delta>0 embeds a green-list watermark."""
    set_seed(seed)
    ids = tokenizer(prompt, return_tensors="pt").input_ids
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1] / temperature
        prev = ids[0, -1].item()
        green = green_list_ids(prev, gamma)
        if delta != 0:
            bias = torch.zeros_like(logits)
            green_idx = torch.tensor(sorted(green))
            bias[green_idx] = delta
            logits = logits + bias
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

PROMPT = "The future of medical research depends on"
watermarked_text   = generate(PROMPT, delta=DELTA)
unwatermarked_text = generate(PROMPT, delta=0.0)

print("WATERMARKED:\n", watermarked_text, "\n")
print("UNWATERMARKED:\n", unwatermarked_text)


> 👀 Notice the watermarked text is still fluent — the bias is gentle. That's the whole point: **stealth** (a reader can't tell) plus **detectability** (a statistical test can).


## 5. Detect the Watermark: A Binomial z-Test

Re-tokenize the text, recompute the green list from each previous token, count how many tokens are green, and test against the null hypothesis "green fraction = $\gamma$".


In [ ]:
def detect(text, gamma=GAMMA):
    """Return green count, total, green fraction, z-score, and p-value."""
    ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()
    green_hits, total = 0, 0
    for i in range(1, len(ids)):
        green = green_list_ids(ids[i - 1], gamma)
        green_hits += ids[i] in green
        total += 1
    frac = green_hits / total
    z = (green_hits - gamma * total) / np.sqrt(total * gamma * (1 - gamma))
    p = stats.norm.sf(z)   # one-sided: P(Z >= z) under H0
    return dict(green=green_hits, total=total, frac=frac, z=z, p=p)

print("Watermarked  :", {k: round(v, 4) if isinstance(v, float) else v for k, v in detect(watermarked_text).items()})
print("Unwatermarked:", {k: round(v, 4) if isinstance(v, float) else v for k, v in detect(unwatermarked_text).items()})


The watermarked text should show a green fraction well above 0.5 and a **large z** (small p-value); the unwatermarked text should sit near 0.5 with $z \approx 0$. The usual decision threshold is $z > 4$ (p ≈ 3×10⁻⁵) — very few false positives.


### ✏️ Exercise 5.1

Run `detect` on a paragraph of **your own human-written text** (paste it below). Is z near 0, as the theory predicts for non-watermarked text? What does a near-zero z tell a platform?


In [ ]:
human_text = """Paste a few sentences you wrote yourself here."""
# print(detect(human_text))


## 6. How Strong Should the Watermark Be? Sweep δ

Larger $\delta$ ⇒ easier detection but more distortion of the text. Let's quantify the trade-off.


In [ ]:
deltas = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0]
z_scores = []
for d in deltas:
    txt = generate(PROMPT, delta=d, max_new_tokens=80)
    z_scores.append(detect(txt)["z"])

plt.figure(figsize=(8, 5))
plt.plot(deltas, z_scores, "o-", color="teal")
plt.axhline(4.0, color="red", ls="--", label="detection threshold z=4")
plt.xlabel("watermark strength δ"); plt.ylabel("detection z-score")
plt.title("Detectability vs. watermark strength"); plt.legend(); plt.grid(alpha=0.3); plt.show()


## 7. False Positives: Testing Un-watermarked Text

A good detector should *rarely* flag genuine human/AI text that carries no watermark. We generate many unwatermarked samples and look at the z-score distribution — it should be centered at 0 with ~5% above z=1.64 (the α=0.05 line), matching the null.


In [ ]:
prompts = [
    "The weather today is", "Scientists recently discovered", "In the city of",
    "My favorite book is about", "The stock market",
    "Doctors recommend that patients", "The history of medicine",
    "A balanced diet includes", "Climate change affects", "The new policy will",
]
null_z = [detect(generate(p, delta=0.0, max_new_tokens=60, seed=i))["z"] for i, p in enumerate(prompts)]

plt.figure(figsize=(8, 4))
plt.hist(null_z, bins=8, color="lightgray", edgecolor="k")
plt.axvline(1.64, color="orange", ls="--", label="α=0.05 (z=1.64)")
plt.axvline(4.0, color="red", ls="--", label="detection threshold z=4")
plt.title("z-scores of UN-watermarked text (should cluster near 0)")
plt.xlabel("z-score"); plt.legend(); plt.show()

print(f"Mean null z = {np.mean(null_z):.2f} (expected ≈ 0)")
print(f"False positives at z>4: {sum(z > 4 for z in null_z)}/{len(null_z)}")


## 8. Robustness: Can Editing Remove the Watermark?

Adversaries paraphrase or delete words to erase the signal. Let's simulate **random word deletion** and see how detection degrades.


In [ ]:
def delete_words(text, fraction, seed=0):
    words = text.split()
    rng = np.random.default_rng(seed)
    keep = rng.random(len(words)) > fraction
    return " ".join(w for w, k in zip(words, keep) if k)

base = generate(PROMPT, delta=DELTA, max_new_tokens=120)
fractions = [0.0, 0.1, 0.2, 0.3, 0.5]
z_after = [detect(delete_words(base, f))["z"] for f in fractions]

plt.figure(figsize=(8, 5))
plt.plot([int(f*100) for f in fractions], z_after, "s-", color="purple")
plt.axhline(4.0, color="red", ls="--", label="threshold z=4")
plt.xlabel("% of words deleted"); plt.ylabel("detection z-score")
plt.title("Watermark robustness under word-deletion attack"); plt.legend(); plt.grid(alpha=0.3); plt.show()


### ✏️ Exercise 8.1

At what deletion percentage does z drop below the threshold? What does this imply for the "**attribution**" outcome on slide 357 — can a platform still make the case after heavy editing?


## 9. 🏆 Challenge Exercises

**Challenge A — Synonym attack.** Replace 15% of words with synonyms (use `nltk.corpus.wordnet`). Is it more or less effective at removing the watermark than deletion? Why?

**Challenge B — ROC curve.** Generate 20 watermarked + 20 unwatermarked texts. Sweep the z-threshold and plot true-positive vs. false-positive rate. Report the AUC.

**Challenge C — Context-window seeding.** Our green list is seeded by a single previous token (h=1). Seed it instead by the previous **two** tokens. How does that affect robustness to single-word edits?


In [ ]:
# Challenge workspace:


## 10. Discussion Questions (slides 340–341)

1. Watermarking trades **stealth vs. detectability vs. robustness**. Which matters most for fighting AI-generated fake news (slide 357)?
2. Detection rests on a binomial test. What is the null hypothesis, and what real-world event is a false positive?
3. **Consent & transparency (slide 341):** should users be told their outputs are watermarked? Argue both sides.
4. Detection needs the model owner's secret green-list scheme. What are the limits of watermarking when the generator is unknown or open-source?


## Key Takeaways

- Watermarking embeds a **statistical signal** in token choices; detection is a **classical binomial/z-test** — the same inference machinery you already know.
- Stronger bias δ ⇒ easier detection but more text distortion: a tunable trade-off.
- The null distribution of z is centered at 0, giving a controllable, principled **false-positive rate**.
- Watermarks are **not robust** to heavy paraphrasing/deletion — an active research frontier (slide 345).

## References

- Kirchenbauer et al. (2023), *A Watermark for Large Language Models* — https://arxiv.org/abs/2301.10226
- Kirchenbauer et al. (2023), *On the Reliability of Watermarks* — https://arxiv.org/abs/2306.04634
